# Listenbrainz data enrichment

Complete Listenbrainz dataset with ISRCs & genres

## Import dependencies and setup musicbrainz api client

In [ ]:

import musicbrainzngs as mb
from musicbrainzngs import WebServiceError
import pandas as pd
from pathlib import Path
import gc
import json
import time
import urllib.error

# MusicBrainz client setup (user agent must be set to avoid being blocked by the API)
mb.set_useragent('NextTrack', '0.1', 'mlocht@outlook.com')

## Example API usages

In [ ]:
# Example call to fetch release data with genres 
release_id = '82082eb8-19a0-4957-9a99-e8fda1bb6b88'
release = None
try:
    result = mb.get_release_by_id(release_id, includes=['recordings', 'isrcs', 'tags'])
except WebServiceError as exc:
    print(f'Web service error: {exc}')
else:
    release = result.get('release', {})
    
print( release)

{'id': '82082eb8-19a0-4957-9a99-e8fda1bb6b88', 'title': 'BRAT', 'status': 'Official', 'quality': 'normal', 'packaging': 'Jewel Case', 'text-representation': {'language': 'eng', 'script': 'Latn'}, 'date': '2024-06-07', 'country': 'US', 'release-event-list': [{'date': '2024-06-07', 'area': {'id': '489ce91b-6658-3307-9877-795b68554c98', 'name': 'United States', 'sort-name': 'United States', 'iso-3166-1-code-list': ['US']}}], 'release-event-count': 1, 'barcode': '075678611698', 'cover-art-archive': {'artwork': 'false', 'count': '0', 'front': 'false', 'back': 'false'}, 'medium-list': [{'position': '1', 'format': 'CD', 'track-list': [{'id': 'a137bda8-1023-4d75-9885-79cc177dcc59', 'position': '1', 'number': '1', 'length': '133800', 'recording': {'id': '9e687f51-1fa6-4374-b9e0-2e3df630bcd5', 'title': '360', 'length': '133805', 'disambiguation': 'explicit', 'isrc-list': ['USAT22401341'], 'isrc-count': 1, 'tag-list': [{'count': '1', 'name': 'bubblegum bass'}, {'count': '1', 'name': 'dance-pop'},

In [4]:
# Extract recording with mbid from the release
recording_mbid = "93e4875c-8a06-4b7d-96cb-8cb27f142c7d"
release_recordings = release.get('medium-list', [])[0].get('track-list', [])
recording = next((rec for rec in release_recordings if rec.get('recording', {}).get('id') == recording_mbid), None)

# extract genres (tags) from the recording
genres = recording.get('recording', {}).get('tag-list', [])
#format genres as a list of genre names
genres = [tag.get('name') for tag in genres]
print(genres)

# extract recording length and ISRCs
recording_length = recording.get('recording', {}).get('length')
isrcs = recording.get('recording', {}).get('isrc-list', [])[0]
print(f"Recording length: {recording_length} ms")
print(f"ISRCs: {isrcs}")

['dance-pop', 'electro house', 'electronic', 'electropop', 'pop']
Recording length: 164000 ms
ISRCs: USAT22313737


## Load preprocessed data

In [ ]:
## Load preprocessed dataset
df01 = pd.read_parquet('original_files/01_clean.parquet')
df02 = pd.read_parquet('original_files/02_clean.parquet')
df03 = pd.read_parquet('original_files/03_clean.parquet')
df = pd.concat([df01, df02, df03], ignore_index=True)

In the following section, a subset df of df is created, that is grouped by unique release_mbid, and contains only unique recording_mbid

In [6]:
df_release_recordings_subset = (
    df[["release_mbid", "recording_mbid"]]
    .dropna(subset=["release_mbid", "recording_mbid"])
    .drop_duplicates()
    .groupby("release_mbid", as_index=False)
    .agg({"recording_mbid": list})
)

df_release_recordings_subset.head()


,release_mbid,recording_mbid
0,00001e72-9d93-4a6a-a45f-cd988f5c161c,[4295492b-1654-4f84-a6c5-661ec152a459]
1,0001c60c-d137-4a61-ba23-e2d65d326dc3,[7434500c-cc91-4b71-905d-24d4b3ccbe2f]
2,0004ec35-ed6d-4325-bd31-f007150d0aba,[7e56c4d1-5cb4-444b-a4cb-3a4e3554152f]
3,00084dc8-3881-4bd9-8f8d-d36157bdf665,"[e264d4f7-a320-4c5a-85f7-5db370e01b9e, 446fc06..."
4,000aba77-8e09-46a9-879e-49231087f201,[044afc05-8125-4daf-b4e8-4caf1b8a54c7]


In [7]:
df_release_recordings_subset.count()

release_mbid      40678
recording_mbid    40678
dtype: int64

In the following section, a second subset of df is created, that contains only the unique recording_mbid values, to this column genres, ISRC, duration are also added; these columns can be nullable. This subset is also saved to CSV.

In [ ]:
recording_metadata_cols = ["recording_mbid", "genres", "ISRC", "duration"]
available_cols = [col for col in recording_metadata_cols if col in df.columns]
output_csv_path = "output/recordings_subset.csv"

if "recording_mbid" not in available_cols:
    raise ValueError("df must contain recording_mbid before building df_recordings_subset")

df_recordings_subset = (
    df[available_cols]
    .dropna(subset=["recording_mbid"])
    .drop_duplicates(subset=["recording_mbid"], keep="first")
    .copy()
)

for col in ["genres", "ISRC", "duration"]:
    if col not in df_recordings_subset.columns:
        df_recordings_subset[col] = pd.NA

df_recordings_subset = df_recordings_subset[recording_metadata_cols]

# The CSV is created only once, to avoid overwriting if the notebook is re-run
from pathlib import Path
if not Path(output_csv_path).exists():
    df_recordings_subset.to_csv(output_csv_path, index=False)

df_recordings_subset.head()


,recording_mbid,genres,ISRC,duration
0,ccf7a8eb-54c0-44d6-961b-be51481c8f1e,<NA>,<NA>,<NA>
1,8257b080-efaa-4b7d-80c6-affaf6875b13,<NA>,<NA>,<NA>
2,50bb6567-e564-46ce-b75f-bfd169503d91,<NA>,<NA>,<NA>
3,a7cb9440-86df-4bff-922e-9ca3a662e331,<NA>,<NA>,<NA>
7,462d8ee7-8084-4f20-b5a2-6b6264922775,<NA>,<NA>,<NA>


In [9]:
df_recordings_subset.count()

recording_mbid    118950
genres                 0
ISRC                   0
duration               0
dtype: int64

## Batch MusicBrainz enrichment with resume + retries

The musicbrainz API is called in batches. The API allows for 300 calls per second globally, using batches avoids saturating the API with calls in peak periods. It also allows to check that the responses returned by the API are ok.

In [ ]:

# Batch configuration
BATCH_RELEASE_LIMIT = 100  # number of release_mbid rows to process in a batch
MAX_RETRIES = 5
RETRY_DELAY_SECONDS = 2
SERVICE_UNAVAILABLE_DELAY_SECONDS = 20  # strong backoff for 503 to avoid hitting MusicBrainz rate limits
RESUME_FROM_STATE = True # when false, start from scratch, when true resume from the last successful API call based on the state file

RATE_LIMIT_INTERVAL_SECONDS = 3.5
RATE_LIMIT_REQUESTS = 1
mb.set_rate_limit(RATE_LIMIT_INTERVAL_SECONDS, RATE_LIMIT_REQUESTS)

RECORDING_COLUMNS = ["recording_mbid", "genres", "ISRC", "duration"]
ENRICHMENT_COLUMNS = ["genres", "ISRC", "duration"]
OUTPUT_CSV_PATH = Path("output/recordings_subset.csv")
STATE_PATH = Path("output/release_enrichment_state.json")

class ReleaseNotFoundError(Exception):
    pass

def _status_code_from_exc(exc):
    cause = getattr(exc, "cause", None)
    code = getattr(cause, "code", None)
    if code is not None:
        return code

    msg = str(exc)
    if "404" in msg:
        return 404
    if "503" in msg:
        return 503
    return None


def is_rate_limited_error(exc):
    return _status_code_from_exc(exc) == 503


def is_not_found_error(exc):
    return _status_code_from_exc(exc) == 404


def is_connection_reset_error(exc):
    text = str(exc)
    return "Errno 54" in text or "Connection reset by peer" in text


def load_progress_state(path):
    if not path.exists():
        return {"last_successful_release_mbid": None}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_progress_state(path, release_mbid):
    state = {
        "last_successful_release_mbid": release_mbid,
        "updated_at_epoch": time.time(),
    }
    with path.open("w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=True, indent=2)


def fetch_release_with_retry(release_mbid):
    attempt = 1
    while attempt <= MAX_RETRIES:
        try:
            result = mb.get_release_by_id(
                release_mbid,
                includes=["recordings", "isrcs", "tags"],
            )
            return result.get("release", {})
        except WebServiceError as exc:
            if is_not_found_error(exc):
                raise ReleaseNotFoundError(f"Release not found: {release_mbid}")
            if is_rate_limited_error(exc):
                delay = SERVICE_UNAVAILABLE_DELAY_SECONDS * attempt
                print(f"[503] {release_mbid} attempt {attempt}/{MAX_RETRIES}. Waiting {delay}s...")
            else:
                delay = RETRY_DELAY_SECONDS * attempt
                print(f"[retry] {release_mbid} attempt {attempt}/{MAX_RETRIES}: {exc}. Waiting {delay}s...")
            time.sleep(delay)
            attempt += 1
        except urllib.error.URLError as exc:
            delay = RETRY_DELAY_SECONDS * attempt
            if is_connection_reset_error(exc):
                print(f"[conn-reset] {release_mbid} attempt {attempt}/{MAX_RETRIES}: {exc}. Waiting {delay}s...")
            else:
                print(f"[url-error] {release_mbid} attempt {attempt}/{MAX_RETRIES}: {exc}. Waiting {delay}s...")
            time.sleep(delay)
            attempt += 1

    raise RuntimeError(f"Failed to fetch release after {MAX_RETRIES} attempts: {release_mbid}")


def extract_recording_metadata_from_release(release_payload):
    metadata_by_recording = {}
    for medium in release_payload.get("medium-list", []) or []:
        for track in medium.get("track-list", []) or []:
            recording = track.get("recording", {}) or {}
            recording_mbid = recording.get("id")
            if not recording_mbid:
                continue

            genres = sorted(
                {
                    tag.get("name")
                    for tag in (recording.get("tag-list", []) or [])
                    if tag.get("name")
                }
            )
            isrc_values = recording.get("isrc-list", []) or []
            duration_value = recording.get("length")

            metadata_by_recording[recording_mbid] = {
                "genres": genres if genres else pd.NA,
                "ISRC": isrc_values[0] if isrc_values else pd.NA,
                "duration": duration_value if duration_value is not None else pd.NA,
            }

    return metadata_by_recording


def filter_to_expected_recordings(metadata_by_recording, expected_recording_mbids):
    if not expected_recording_mbids:
        return {}
    expected_recording_mbids = {str(mbid) for mbid in expected_recording_mbids}
    return {
        recording_mbid: values
        for recording_mbid, values in metadata_by_recording.items()
        if str(recording_mbid) in expected_recording_mbids
    }


def ensure_recording_columns(df_subset):
    df_subset = df_subset.copy()
    for col in RECORDING_COLUMNS:
        if col not in df_subset.columns:
            df_subset[col] = pd.NA
    return df_subset[RECORDING_COLUMNS]


def order_recordings_by_enrichment(df_subset):
    work = df_subset.copy()
    work["enriched_fields_count"] = work[ENRICHMENT_COLUMNS].notna().sum(axis=1)
    work = work.sort_values(
        ["enriched_fields_count", "recording_mbid"],
        ascending=[False, True],
        kind="stable",
    )
    return work.drop(columns=["enriched_fields_count"]).reset_index(drop=True)


def save_checkpoint(df_subset, csv_path):
    df_subset.to_csv(csv_path, index=False)
    return df_subset


def count_release_recordings(recording_mbids):
    return len(recording_mbids) if isinstance(recording_mbids, list) else 0


def resolve_start_index(ordered_releases, state, resume_enabled):
    if not resume_enabled:
        return 0

    last_release_mbid = state.get("last_successful_release_mbid")
    if not last_release_mbid:
        return 0

    matches = ordered_releases.index[ordered_releases["release_mbid"] == last_release_mbid].tolist()
    return matches[0] + 1 if matches else 0


def summarize_enrichment(df_subset):
    total_rows = int(len(df_subset))
    isrc_match_count = int(df_subset["ISRC"].notna().sum())

    incomplete_mask = df_subset[ENRICHMENT_COLUMNS].isna().all(axis=1)
    incomplete_rows = int(incomplete_mask.sum())
    metadata_found_count = int(total_rows - incomplete_rows)
    enrichment_completed_pct = (metadata_found_count / total_rows * 100.0) if total_rows else 0.0

    return {
        "isrc_match_count": isrc_match_count,
        "total_rows": total_rows,
        "incomplete_rows": incomplete_rows,
        "metadata_found_count": metadata_found_count,
        "enrichment_completed_pct": enrichment_completed_pct,
    }


if OUTPUT_CSV_PATH.exists():
    recordings_subset_df = pd.read_csv(OUTPUT_CSV_PATH)
elif "df_recordings_subset" in globals():
    recordings_subset_df = df_recordings_subset.copy()
else:
    raise ValueError("df_recordings_subset not found in memory and CSV not found at output/recordings_subset.csv")

recordings_subset_df = ensure_recording_columns(recordings_subset_df)
recordings_subset_df = recordings_subset_df.drop_duplicates(subset=["recording_mbid"], keep="first").copy()
recordings_subset_df["recording_mbid"] = recordings_subset_df["recording_mbid"].astype(str)

if "df" in globals() and "recording_mbid" in df.columns:
    canonical_recording_mbids = set(df["recording_mbid"].dropna().astype(str).unique())
    pre_filter_count = len(recordings_subset_df)
    recordings_subset_df = recordings_subset_df[
        recordings_subset_df["recording_mbid"].isin(canonical_recording_mbids)
    ].copy()
    removed_count = pre_filter_count - len(recordings_subset_df)
else:
    canonical_recording_mbids = set(recordings_subset_df["recording_mbid"])

initial_recording_row_count = len(recordings_subset_df)
allowed_recording_mbids = canonical_recording_mbids

row_index_by_mbid = {
    str(mbid): idx
    for idx, mbid in recordings_subset_df["recording_mbid"].items()
}

ordered_releases_df = (
    df_release_recordings_subset
    .copy()
    .assign(recording_count=lambda d: d["recording_mbid"].apply(count_release_recordings))
    .sort_values("recording_count", ascending=False)
    .reset_index(drop=True)
)

progress_state = load_progress_state(STATE_PATH)
start_idx = resolve_start_index(ordered_releases_df, progress_state, RESUME_FROM_STATE)
release_batch_df = ordered_releases_df.iloc[start_idx:start_idx + BATCH_RELEASE_LIMIT]

print(f"Processing {len(release_batch_df)} release(s) starting at index {start_idx}.")

processed_release_count = 0
skipped_not_found_count = 0

for _, release_row in release_batch_df.iterrows():
    release_mbid = release_row["release_mbid"]
    expected_count = int(release_row["recording_count"])

   
    release_payload = fetch_release_with_retry(release_mbid)
 

    metadata_by_recording = extract_recording_metadata_from_release(release_payload)

    expected_recording_mbids = (
        set(release_row["recording_mbid"])
        if isinstance(release_row["recording_mbid"], list)
        else set()
    )
    metadata_by_recording = filter_to_expected_recordings(metadata_by_recording, expected_recording_mbids)
    metadata_by_recording = {
        recording_mbid: values
        for recording_mbid, values in metadata_by_recording.items()
        if str(recording_mbid) in allowed_recording_mbids
    }

    for recording_mbid, values in metadata_by_recording.items():
        row_idx = row_index_by_mbid.get(str(recording_mbid))
        if row_idx is None:
            continue
        for col, value in values.items():
            recordings_subset_df.at[row_idx, col] = value

    if len(recordings_subset_df) != initial_recording_row_count:
        raise RuntimeError(
            f"Row count changed unexpectedly: {len(recordings_subset_df)} != {initial_recording_row_count}"
        )

    processed_release_count += 1
    print(f"Saved progress after {release_mbid}. Updated rows: {len(metadata_by_recording)}")

    del release_payload, metadata_by_recording


if len(recordings_subset_df) != initial_recording_row_count:
    raise RuntimeError(
        f"Final row count changed unexpectedly: {len(recordings_subset_df)} != {initial_recording_row_count}"
    )

recordings_subset_df = save_checkpoint(recordings_subset_df, OUTPUT_CSV_PATH)
summary = summarize_enrichment(recordings_subset_df)

print(f"Done. Releases processed in this batch: {processed_release_count}")
print(f"Skipped not found releases (404): {skipped_not_found_count}")
print(
    f"Found metadata for {summary['metadata_found_count']} recordings, "
    f"of which {summary['isrc_match_count']} have ISRCs."
)
print(f"Enrichment {summary['enrichment_completed_pct']:.2f}% completed.")

df_recordings_subset = recordings_subset_df
df_recordings_subset.head()


Processing 100 release(s) starting at index 5517.
Saved progress after a47ad89f-d224-3082-9aa9-61783d724cdc. Updated rows: 7
Saved progress after f8330050-46c8-4402-b6bf-cba9b55b35ff. Updated rows: 7
Saved progress after 852da111-2333-30ee-a4a2-b2fd9c5af9bd. Updated rows: 7
Saved progress after 941c5882-663a-42ae-a519-e1d4bbc27299. Updated rows: 7
Saved progress after 9735783d-2807-4afd-82f8-2cf63b28ed50. Updated rows: 7
Saved progress after a6428eae-2ba5-47a3-8ffd-6e60509334fd. Updated rows: 7
Saved progress after 9791d0d9-99b1-4967-a33f-c07564d3e7b2. Updated rows: 5
Saved progress after 0946a9c4-d208-4d1b-aa39-887bc0454d33. Updated rows: 6
Saved progress after 094224c6-c99f-4195-86bd-624ca0d438fd. Updated rows: 7
Saved progress after f7b1f0ff-3580-4ad9-ba56-c9b6a4122940. Updated rows: 7
Saved progress after 976715b6-7cdd-4008-a500-f83dd2a7f2b7. Updated rows: 7
Saved progress after f7a3ebf1-1d85-4126-9b04-49509657ec95. Updated rows: 7
Saved progress after 0d3cda75-733b-4510-8b42-2a3c4

,recording_mbid,genres,ISRC,duration
0,0003dd36-b4d2-4216-a37e-b110f6882ecb,"['alternative rock', 'ambient', 'electronic', ...",GBDCA9900070,491000.0
1,00047577-1669-4da1-9aa5-7d7b7923ebdf,"['alternative rock', 'emo', 'indie', 'indie ro...",US3R49900018,163053.0
2,0006fc51-adb4-4417-b8fb-7b954b853923,"['2-step', 'ambient', 'dubstep', 'electronic',...",GBLZC0500002,301506.0
3,0006ff85-f08f-4c5a-844d-ce0ec70ad670,"['billboard hot 100', 'billboard hot 100 2025'...",USUG12408498,233000.0
4,000dbffe-59b2-42ba-9458-c8989dccaeb9,"['electronic', 'electronica', 'electropop', 'e...",GBAJH0900116,211266.0
